In [1]:
import pandas as pd
import re
import sys
from pathlib import Path

In [2]:
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')
g = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t') 

In [3]:
sys.path.append("..")
from src.normalization import extract_geo_layers

In [6]:
def find_optimal_threshold(s1, s2, s3, thresholds=[10000, 12000, 15000, 20000]):
    print("Combining datasets for global threshold calculation...")
    
    combined = pd.concat([
        s1[['country', 'business_address']], 
        s2[['country', 'business_address']], 
        s3[['country', 'business_address']]
    ]).dropna(subset=['business_address']).copy()
    
    print("Extracting base layers (this takes a moment)...")
    geo_data = combined.apply(lambda row: extract_geo_layers(row['business_address'], row['country']), axis=1)
    geo_df = pd.DataFrame(geo_data.tolist())
    
    def assign_base(row):
        if row['pin']: return f"{row['country']}_{row['pin']}"
        elif row['chunk_1']: return f"{row['country']}_{row['chunk_1']}"
        return f"{row['country']}_unknown"
        
    print("Assigning base keys...")
    sim_df = pd.DataFrame({
        'base_key': geo_df.apply(assign_base, axis=1),
        'chunk_1': geo_df['chunk_1'],
        'chunk_2': geo_df['chunk_2'],
        'country': geo_df['country']
    })
    
    base_counts = sim_df['base_key'].value_counts()
    
    print("\n" + "="*30)
    print("SWEEPING THRESHOLDS")
    print("="*30)
    
    for thresh in thresholds:
        massive_keys = set(base_counts[base_counts > thresh].index)
        
        def simulate_split(row):
            key = row['base_key']
            if key in massive_keys and not re.search(r'\d{5,6}$', key):
                chunk_2 = row['chunk_2']
                if chunk_2:
                    return f"{row['country']}_{chunk_2}_{row['chunk_1']}"
            return key
            
        final_keys = sim_df.apply(simulate_split, axis=1)
        final_counts = final_keys.value_counts()
        
        print(f"\n[ Threshold: {thresh} ]")
        print(f"-> Massive Blocks Split: {len(massive_keys)}")
        print(f"-> Max Block Size: {final_counts.max()} records")
        print(f"-> No of Blocks: {len(final_counts)}")
        print("-> Top 3 Largest Blocks Post-Split:")
        print(final_counts.head(3).to_string())

# Execute the sweep on your raw sample dataframes
find_optimal_threshold(s1, s2, s3)

Combining datasets for global threshold calculation...
Extracting base layers (this takes a moment)...
Assigning base keys...

SWEEPING THRESHOLDS

[ Threshold: 10000 ]
-> Massive Blocks Split: 128
-> Max Block Size: 115504 records
-> No of Blocks: 1298807
-> Top 3 Largest Blocks Post-Split:
india_mumbai_maharashtra     115504
india_bangalore_karnataka     96886
india_new_delhi_delhi         93253

[ Threshold: 12000 ]
-> Massive Blocks Split: 124
-> Max Block Size: 115504 records
-> No of Blocks: 1292389
-> Top 3 Largest Blocks Post-Split:
india_mumbai_maharashtra     115504
india_bangalore_karnataka     96886
india_new_delhi_delhi         93253

[ Threshold: 15000 ]
-> Massive Blocks Split: 117
-> Max Block Size: 115504 records
-> No of Blocks: 1271269
-> Top 3 Largest Blocks Post-Split:
india_mumbai_maharashtra     115504
india_bangalore_karnataka     96886
india_new_delhi_delhi         93253

[ Threshold: 20000 ]
-> Massive Blocks Split: 106
-> Max Block Size: 115504 records
-> No 